## Torch

In [25]:
import torch
from torchvision import transforms


normalize = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
def normalize_transform(image):
    return normalize(image)


def make_grid(N, iH, iW):
    grid_x = torch.linspace(-1.0, 1.0, iW).view(1, 1, iW, 1).expand(N, iH, -1, -1)
    grid_y = torch.linspace(-1.0, 1.0, iH).view(1, iH, 1, 1).expand(N, -1, iW, -1)
    grid = torch.cat([grid_x, grid_y], 3)
    return grid

In [26]:
from PIL import Image

sample_image = Image.fromarray((torch.rand(256, 256, 3) * 255).byte().numpy())

%timeit normalize_transform(sample_image)

%timeit make_grid(4, 256, 256)

799 µs ± 10.2 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
1.02 ms ± 145 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


## TorchJIT

In [27]:
import torch
from torchvision import transforms


@torch.jit.script
def normalize_transform(image: torch.Tensor) -> torch.Tensor:
    image = image * 1.0 / 255
    return (image - 0.5) / 0.5

@torch.jit.script
def make_grid(N: int, iH: int, iW: int) -> torch.Tensor:
    grid_x = torch.linspace(-1.0, 1.0, iW).view(1, 1, iW, 1).expand(N, iH, -1, -1)
    grid_y = torch.linspace(-1.0, 1.0, iH).view(1, iH, 1, 1).expand(N, -1, iW, -1)
    grid = torch.cat([grid_x, grid_y], 3)
    return grid

In [28]:
from PIL import Image

sample_image = (torch.rand(256, 256, 3) * 255).byte()

%timeit normalize_transform(sample_image)

%timeit make_grid(4, 256, 256)

390 µs ± 7.25 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
1.03 ms ± 149 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


## Numba

In [3]:
!pip install numba

In [29]:
import numpy as np
import numba
from numba import prange


@numba.jit(nopython=True, parallel=True)
def make_grid(N, iH, iW):
    grid = np.zeros((iH, iW, 2), dtype=np.float32)
    for y in prange(iH):
        for x in prange(iW):
            grid[y, x, 0] = -1.0 + 2.0 * x / (iW - 1)  # grid_x
            grid[y, x, 1] = -1.0 + 2.0 * y / (iH - 1)  # grid_y
    return grid


@numba.jit(nopython=True, parallel=True)
def normalize_transform(image):
    image = image * 1.0 / 255
    return (image - 0.5) / 0.5

In [30]:
from PIL import Image

sample_image = (torch.rand(256, 256, 3) * 255).byte().numpy()

%timeit normalize_transform(sample_image)

%timeit make_grid(4, 256, 256)

658 µs ± 78 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
418 µs ± 97 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
